In [7]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

In [8]:
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

train = pd.read_csv(
    root / "data" / "raw" / "train.csv"
)

data = train.copy()
data["FamilySize"] = (
    data["SibSp"] + data["Parch"] + 1
)
data["IsAlone"] = (
    data["FamilySize"] == 1
).astype(int)

numerical_features = [
    "Age",
    "Fare",
    "FamilySize",
    "IsAlone",
]
categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
]

features = (
    numerical_features + categorical_features
)
X = data[features]
y = data["Survived"]

numerical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median"),
    ),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent"),
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
    ),
])

preprocessor = ColumnTransformer([
    (
        "numerical",
        numerical_pipeline,
        numerical_features,
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features,
    ),
])

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=1,
    ),
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [9]:
rows = []

for model_name, classifier in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "recall": "recall",
        },
        n_jobs=-1,
    )

    rows.append({
        "model": model_name,
        "accuracy_mean": scores[
            "test_accuracy"
        ].mean(),
        "accuracy_std": scores[
            "test_accuracy"
        ].std(),
        "survivor_recall_mean": scores[
            "test_recall"
        ].mean(),
    })

results = (
    pd.DataFrame(rows)
    .set_index("model")
    .sort_values(
        "accuracy_mean",
        ascending=False,
    )
)

display(results.round(3))

lr = results.loc["Logistic Regression"]
rf = results.loc["Random Forest"]
delta = (
    rf["accuracy_mean"]
    - lr["accuracy_mean"]
)

print(
    f"Accuracy delta (RF - LR): {delta:+.3f}"
)

,accuracy_mean,accuracy_std,survivor_recall_mean
model,,,
Random Forest,0.831,0.015,0.710
Logistic Regression,0.802,0.015,0.702


Accuracy delta (RF - LR): +0.028


## Model comparison decision

- Logistic Regression: accuracy **0.802 ± 0.015**, survivor recall **0.702**.
- Random Forest: accuracy **0.831 ± 0.015**, survivor recall **0.710**.
- Accuracy delta (Random Forest − Logistic Regression): **+0.028**.
- Decision: **Random Forest becomes the model-v2 candidate** because its accuracy improvement exceeds the `+0.005` rule. It also has slightly higher survivor recall with the same score variability.
- This is still a local validation result; Random Forest must beat the Kaggle baseline of **0.76794** before becoming the preferred competition model.